[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t05_sum_formulas.ipynb)

# Ptolemy's table needed sin(A + B): the sum and difference formulas

Hipparchus made the first known table of chords about 140 BCE; it is lost. Around 150 CE, Ptolemy built one in
half-degree steps, to predict the sky. To fill it, he proved what we now call the sine of a sum.

Here the formulas come from one move: **turning the arrow by x, then turning it again by y.**

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Two turns, one after the other

Turn the arrow by x. Now stand on that arrow, as if it were a new horizontal axis, and turn by y more. The arrow ends up at
x + y. The question is: what are its projections?

Seen from the first arrow, the second turn has its own projections: **cos y along the first arrow**, and **sin y at
right angles to it**. Each of those two pieces has its own projections on the real axes. Add them up.

In [ ]:
def two_turns(x_deg=35.0, y_deg=40.0):
    x, y = math.radians(x_deg), math.radians(y_deg)
    u = np.array([math.cos(x), math.sin(x)])                  # the first arrow: the new horizontal axis
    v = np.array([-math.sin(x), math.cos(x)])                 # the new vertical axis, at right angles to it
    p = math.cos(y) * u                                        # the second turn's horizontal projection, along u
    q = p + math.sin(y) * v                                    # then its vertical projection, along v: the final tip
    fig, ax = plt.subplots(figsize=(7.4, 7.4))
    circle_axes(ax, lim=1.3)
    ax.plot([-1.3 * u[0], 1.3 * u[0]], [-1.3 * u[1], 1.3 * u[1]], color=TEAL, lw=1, ls="--")
    ax.plot([-1.3 * v[0], 1.3 * v[0]], [-1.3 * v[1], 1.3 * v[1]], color=TEAL, lw=1, ls="--")
    arrow(ax, x, color=TEAL, lw=2, label="x")
    ax.plot([0, p[0]], [0, p[1]], color=SAGE, lw=6, alpha=0.8, solid_capstyle="butt")
    ax.plot([p[0], q[0]], [p[1], q[1]], color=LILAC, lw=6, alpha=0.8, solid_capstyle="butt")
    arrow(ax, x + y, label="x + y")
    ax.plot([q[0], q[0]], [0, q[1]], color=DIM, ls=":"); ax.plot([0, q[0]], [q[1], q[1]], color=DIM, ls=":")
    ax.text(-1.25, -1.05, f"cos y along the first arrow (sage) = {math.cos(y):.3f}\n"
                          f"sin y across it (lilac) = {math.sin(y):.3f}\n"
                          f"tip: ({q[0]:.3f}, {q[1]:.3f}) = (cos(x+y), sin(x+y))", fontsize=11, va="top")
    ax.set_title(f"turn {x_deg:.0f}°, then {y_deg:.0f}° more"); plt.show()

interact(two_turns, x_deg=FloatSlider(value=35, min=0, max=180, step=5, description="x (°)"),
         y_deg=FloatSlider(value=40, min=0, max=180, step=5, description="y (°)"));

**Notice.** The sage piece is cos y long and points along the first arrow. So it reaches cos y × cos x across, and
cos y × sin x up. The lilac piece is sin y long, at right angles to the first arrow. So it reaches −sin y × sin x
across, and sin y × cos x up. Add the two across parts, then the two up parts:

**cos(x + y) = cos x cos y − sin x sin y**

**sin(x + y) = sin x cos y + cos x sin y**

In [ ]:
def pieces(x_deg=35.0, y_deg=40.0):
    x, y = math.radians(x_deg), math.radians(y_deg)
    print(f"cos x cos y − sin x sin y = {math.cos(x) * math.cos(y) - math.sin(x) * math.sin(y):+.6f}")
    print(f"cos(x + y)                = {math.cos(x + y):+.6f}")
    print(f"sin x cos y + cos x sin y = {math.sin(x) * math.cos(y) + math.cos(x) * math.sin(y):+.6f}")
    print(f"sin(x + y)                = {math.sin(x + y):+.6f}")

interact(pieces, x_deg=FloatSlider(value=35, min=-180, max=180, step=5, description="x (°)"),
         y_deg=FloatSlider(value=40, min=-180, max=180, step=5, description="y (°)"));

## 2 · Differences, for free

Turning by −y is turning back. Since cos(−y) = cos y and sin(−y) = −sin y, put −y in place of y:

**cos(x − y) = cos x cos y + sin x sin y** and **sin(x − y) = sin x cos y − cos x sin y.**

## 3 · Allied angles are the circle's mirrors

Put special values in the formulas and the "allied angle" rules fall out. Better still, each one is a symmetry of the
circle: a mirror or a quarter turn. Pick one and drag x.

In [ ]:
MOVES = {"−x  (mirror in the horizontal axis)": lambda a: -a,
         "π/2 − x  (mirror in the diagonal)": lambda a: math.pi / 2 - a,
         "π/2 + x  (quarter turn)": lambda a: math.pi / 2 + a,
         "π − x  (mirror in the vertical axis)": lambda a: math.pi - a,
         "π + x  (half turn)": lambda a: math.pi + a,
         "2π − x  (full turn back, mirrored)": lambda a: 2 * math.pi - a}
RULES = {"−x  (mirror in the horizontal axis)": "sin(−x) = −sin x,  cos(−x) = cos x",
         "π/2 − x  (mirror in the diagonal)": "sin(π/2 − x) = cos x,  cos(π/2 − x) = sin x",
         "π/2 + x  (quarter turn)": "sin(π/2 + x) = cos x,  cos(π/2 + x) = −sin x",
         "π − x  (mirror in the vertical axis)": "sin(π − x) = sin x,  cos(π − x) = −cos x",
         "π + x  (half turn)": "sin(π + x) = −sin x,  cos(π + x) = −cos x",
         "2π − x  (full turn back, mirrored)": "sin(2π − x) = −sin x,  cos(2π − x) = cos x"}

def mirrors(move="π − x  (mirror in the vertical axis)", x_deg=30.0):
    a = math.radians(x_deg); b = MOVES[move](a)
    fig, ax = plt.subplots(figsize=(6.6, 6.6))
    circle_axes(ax)
    if "diagonal" in move: ax.plot([-1.3, 1.3], [-1.3, 1.3], color=DIM, ls="--", lw=1)
    if "vertical axis" in move: ax.axvline(0, color=DIM, ls="--", lw=2)
    if "horizontal axis" in move or "2π" in move: ax.axhline(0, color=DIM, ls="--", lw=2)
    arrow(ax, a, color=TEAL, label="x"); arrow(ax, b, label=move.split("  ")[0])
    ax.set_title(RULES[move], fontsize=12.5)
    plt.show()

interact(mirrors, move=Dropdown(options=list(MOVES), value="π − x  (mirror in the vertical axis)", description="angle"),
         x_deg=FloatSlider(value=30, min=0, max=90, step=5, description="x (°)"));

**Notice.** In the mirror in the vertical axis (π − x), the vertical projection stays and the horizontal one flips. In the half turn
(π + x), both flip. You never need to memorise these: picture the mirror.

## 4 · tan and cot of a sum

Divide the sine formula by the cosine formula, then divide top and bottom by cos x cos y:

**tan(x + y) = (tan x + tan y) ÷ (1 − tan x tan y)**, and with −y, **tan(x − y) = (tan x − tan y) ÷ (1 + tan x tan y)**.

Dividing by sin x sin y instead gives **cot(x + y) = (cot x cot y − 1) ÷ (cot y + cot x)** and
**cot(x − y) = (cot x cot y + 1) ÷ (cot y − cot x)**.

## 5 · Ptolemy's trick: a table from a few seeds

Ptolemy knew a few exact values from regular shapes: 30°, 36° and 72° (the pentagon), 45° and 60°. With the
difference formulas, those seeds give 6°, 9°, 12° and 15°, then 3°, and then every multiple of 3° by adding.
Here is that table, built only with sums and differences, against a modern calculator.

In [ ]:
SEED = {30: (0.5, math.sqrt(3) / 2), 45: (math.sqrt(2) / 2, math.sqrt(2) / 2), 60: (math.sqrt(3) / 2, 0.5),
        36: (math.sqrt(10 - 2 * math.sqrt(5)) / 4, (1 + math.sqrt(5)) / 4),
        72: (math.sqrt(10 + 2 * math.sqrt(5)) / 4, (math.sqrt(5) - 1) / 4)}
table = dict(SEED)
def diff(a, b):
    (sa, ca), (sb, cb) = table[a], table[b]
    table[a - b] = (sa * cb - ca * sb, ca * cb + sa * sb)
def add(a, b):
    (sa, ca), (sb, cb) = table[a], table[b]
    table[a + b] = (sa * cb + ca * sb, ca * cb - sa * sb)
diff(36, 30); diff(45, 36); diff(72, 60); diff(45, 30); diff(9, 6)          # 6°, 9°, 12°, 15°, 3°
for k in range(2, 31):
    if 3 * k not in table: add(3 * (k - 1), 3)
print(f"{'angle':>6} {'from seeds':>12} {'calculator':>12}")
worst = 0.0
for d in sorted(table):
    if d % 3 == 0 and 0 < d <= 90:
        s = table[d][0]; worst = max(worst, abs(s - math.sin(math.radians(d))))
        if d % 9 == 0 or d in (3, 15, 75): print(f"{d:5d}° {s:12.9f} {math.sin(math.radians(d)):12.9f}")
print(f"every multiple of 3° up to 90°: largest difference {worst:.1e}")
assert worst < 1e-12

**Notice.** Thirty-one angles from five seeds, and they agree with the calculator to the last digit. Ptolemy's table
went further, to every half degree, with a clever estimate for the last step.

## 6 · The identity gallery

The class 11 list in full (NCERT identities 1–13). The two-angle formulas are plotted against x with y fixed at
0.7 radians, and checked at random x and y.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 1500)
Y = 0.7
two = [("cos(x + y) = cos x cos y − sin x sin y", lambda x, y: np.cos(x + y), lambda x, y: np.cos(x) * np.cos(y) - np.sin(x) * np.sin(y)),
       ("cos(x − y) = cos x cos y + sin x sin y", lambda x, y: np.cos(x - y), lambda x, y: np.cos(x) * np.cos(y) + np.sin(x) * np.sin(y)),
       ("sin(x + y) = sin x cos y + cos x sin y", lambda x, y: np.sin(x + y), lambda x, y: np.sin(x) * np.cos(y) + np.cos(x) * np.sin(y)),
       ("sin(x − y) = sin x cos y − cos x sin y", lambda x, y: np.sin(x - y), lambda x, y: np.sin(x) * np.cos(y) - np.cos(x) * np.sin(y))]
fig, axs = plt.subplots(2, 2, figsize=(12, 6.4))
for ax, (name, L, R) in zip(axs.flat, two):
    both_sides(ax, x, L(x, Y), R(x, Y), name + "   (y = 0.7)")
axs[0, 0].legend(loc="lower right"); plt.tight_layout(); plt.show()
for name, L, R in two:
    print(name); check_identity(L, R, two=True)

**The allied angles** (NCERT identities 5, 6 and 9): each is a mirror or a turn of the circle.

In [ ]:
allied = [("sin(π/2 − x) = cos x", lambda x: np.sin(np.pi / 2 - x), np.cos),
          ("cos(π/2 − x) = sin x", lambda x: np.cos(np.pi / 2 - x), np.sin),
          ("sin(π/2 + x) = cos x", lambda x: np.sin(np.pi / 2 + x), np.cos),
          ("cos(π/2 + x) = −sin x", lambda x: np.cos(np.pi / 2 + x), lambda x: -np.sin(x)),
          ("sin(π − x) = sin x", lambda x: np.sin(np.pi - x), np.sin),
          ("cos(π − x) = −cos x", lambda x: np.cos(np.pi - x), lambda x: -np.cos(x)),
          ("sin(π + x) = −sin x", lambda x: np.sin(np.pi + x), lambda x: -np.sin(x)),
          ("cos(π + x) = −cos x", lambda x: np.cos(np.pi + x), lambda x: -np.cos(x)),
          ("sin(2π − x) = −sin x", lambda x: np.sin(2 * np.pi - x), lambda x: -np.sin(x)),
          ("cos(2π − x) = cos x", lambda x: np.cos(2 * np.pi - x), np.cos)]
fig, axs = plt.subplots(5, 2, figsize=(12, 13))
for ax, (name, L, R) in zip(axs.flat, allied):
    both_sides(ax, x, L(x), R(x), name)
plt.tight_layout(); plt.show()
for name, L, R in allied:
    print(name); check_identity(L, R, n=3)

**tan and cot of a sum and a difference** (NCERT identities 10–13). Checked away from the angles where a side is
undefined.

In [ ]:
tan, cot = np.tan, lambda v: 1 / np.tan(v)
ok_t = lambda x, y: (np.abs(np.cos(x)) > 0.1) & (np.abs(np.cos(y)) > 0.1) & (np.abs(np.cos(x + y)) > 0.1) & (np.abs(np.cos(x - y)) > 0.1)
ok_c = lambda x, y: (np.abs(np.sin(x)) > 0.1) & (np.abs(np.sin(y)) > 0.1) & (np.abs(np.sin(x + y)) > 0.1) & (np.abs(np.sin(x - y)) > 0.1)
tc = [("tan(x + y) = (tan x + tan y) ÷ (1 − tan x tan y)", lambda x, y: tan(x + y), lambda x, y: (tan(x) + tan(y)) / (1 - tan(x) * tan(y)), ok_t),
      ("tan(x − y) = (tan x − tan y) ÷ (1 + tan x tan y)", lambda x, y: tan(x - y), lambda x, y: (tan(x) - tan(y)) / (1 + tan(x) * tan(y)), ok_t),
      ("cot(x + y) = (cot x cot y − 1) ÷ (cot y + cot x)", lambda x, y: cot(x + y), lambda x, y: (cot(x) * cot(y) - 1) / (cot(y) + cot(x)), ok_c),
      ("cot(x − y) = (cot x cot y + 1) ÷ (cot y − cot x)", lambda x, y: cot(x - y), lambda x, y: (cot(x) * cot(y) + 1) / (cot(y) - cot(x)), ok_c)]
fig, axs = plt.subplots(2, 2, figsize=(12, 6.4))
for ax, (name, L, R, _) in zip(axs.flat, tc):
    with np.errstate(divide="ignore", invalid="ignore"):
        both_sides(ax, x, L(x, Y), R(x, Y), name, ylim=(-6, 6))
plt.tight_layout(); plt.show()
for name, L, R, ok in tc:
    print(name); check_identity(L, R, two=True, ok=ok)

## 7 · On paper: the CBSE way

**Example 1.** Find sin 15°.

1. Write 15° as a difference of angles you know: 15° = 45° − 30°.
2. sin(45° − 30°) = sin 45° cos 30° − cos 45° sin 30° = (√2/2)(√3/2) − (√2/2)(1/2).
3. **sin 15° = (√6 − √2) ÷ 4.**

In [ ]:
print(f"(√6 − √2)/4 = {(math.sqrt(6) - math.sqrt(2)) / 4:.9f},  sin 15° = {math.sin(math.radians(15)):.9f}")

**Example 2.** Find tan(13π/12).

1. 13π/12 = π + π/12, and tan repeats every π, so tan(13π/12) = tan(π/12) = tan(45° − 30°).
2. tan(45° − 30°) = (1 − 1/√3) ÷ (1 + 1/√3) = (√3 − 1) ÷ (√3 + 1).
3. Multiply top and bottom by (√3 − 1): (4 − 2√3) ÷ 2. **tan(13π/12) = 2 − √3.**

In [ ]:
print(f"2 − √3 = {2 - math.sqrt(3):.9f},  tan(13π/12) = {math.tan(13 * math.pi / 12):.9f}")

**Try it.**
1. Find cos 75°.
2. Show that cos(π/4 + x) + cos(π/4 − x) = √2 cos x.

<details><summary>Answers</summary>

1. cos(45° + 30°) = cos 45° cos 30° − sin 45° sin 30° = (√6 − √2) ÷ 4.
2. Add the two formulas: the sin terms cancel, leaving 2 cos(π/4) cos x = 2 (√2/2) cos x = √2 cos x.

</details>

In [ ]:
xx = np.linspace(-3, 3, 7)
assert np.allclose(np.cos(np.pi / 4 + xx) + np.cos(np.pi / 4 - xx), np.sqrt(2) * np.cos(xx), atol=1e-12)
print("try-it 2 checked at", len(xx), "angles")

## Check yourself
1. Find sin 75°.
2. What is sin(π − x), and which mirror explains it?
3. Why does tan(45° + 45°) not exist, according to the formula?

<details><summary>Answers</summary>

1. sin(45° + 30°) = (√6 + √2) ÷ 4.
2. sin x: the mirror in the vertical axis keeps the vertical projection.
3. The bottom is 1 − tan 45° tan 45° = 1 − 1 = 0: the line at 90° never meets the line x = 1.

</details>

## Sources
MacTutor History of Mathematics (St Andrews): "Ptolemy"; "Hipparchus"; "Trigonometric functions"; Strick, notes on
Ptolemy's chord table. NCERT class 11, chapter 3 (identities 1–13). CBSE Mathematics syllabus 2026–27. All retrieved
2026-10-21.